# 2D FP32 Training Export v0.5

This notebook extends v0.4 by scanning `./training-data` (excluding `shards`) and matching directory names against `./datasets`. Only unmatched dataset directories are selected for export.

Set `PROCESS = False` for a dry-run report of files that would be processed.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
import ipywidgets as widgets
from ipyfilechooser import FileChooser


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'preprocessing' / 'export_2d_training.py').exists():
            return candidate
    raise RuntimeError('Could not find repo root containing preprocessing/export_2d_training.py')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from preprocessing.export_2d_training import ExportConfig, export_directory

display(Markdown(f'Using repo root: `{REPO_ROOT}`'))


In [ ]:
DEFAULT_DATASETS_DIR = REPO_ROOT / 'datasets'
DEFAULT_TRAINING_DATA_DIR = REPO_ROOT / 'training-data'

BACKEND = 'torch-cuda'  # Change to 'cpu' for debug fallback
SKIP_EXISTING = True
PROCESS = True  # False = dry-run report only (no export)

# Optional filters / smoke-test controls
TARGET_DATASET_DIR_NAMES = None  # Example: ['+6db-fan-normal']
MAX_FILES_PER_DIRECTORY = None    # Example: 5

# Feature / export configuration
TARGET_SR = 16000
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = 'hann'
NUM_BANDS = 96
FMIN = 50.0
FMAX = 8000.0
LOG_POWER_REFERENCE = 1.0
EPSILON = 1e-8
ENERGY_BINS = 24
LOW_ENERGY_FLOOR = 0.0
MIN_ACTIVE_BANDS_PER_FRAME = 0
WINDOW_FRAMES = 64
STRIDE_FRAMES = 32


## Select Datasets And Training-Data Roots
The notebook scans directory names under `training-data` (excluding paths under `shards`) and compares them to dataset directory names under `datasets`.


In [ ]:
def _chooser_start_dir(path: Path) -> Path:
    if path.exists():
        return path
    for candidate in [path.parent, REPO_ROOT, Path.cwd()]:
        if candidate.exists():
            return candidate
    return Path.cwd()

datasets_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_DATASETS_DIR)))
datasets_dir_chooser.title = '<b>Datasets root directory</b>'
datasets_dir_chooser.show_only_dirs = True
datasets_dir_chooser.use_dir_icons = True
datasets_dir_chooser.select_default = True

training_data_dir_chooser = FileChooser(path=str(_chooser_start_dir(DEFAULT_TRAINING_DATA_DIR)))
training_data_dir_chooser.title = '<b>Training-data root directory</b>'
training_data_dir_chooser.show_only_dirs = True
training_data_dir_chooser.use_dir_icons = True
training_data_dir_chooser.select_default = True

display(
    widgets.HBox([
        widgets.VBox([datasets_dir_chooser]),
        widgets.VBox([training_data_dir_chooser]),
    ])
)


In [ ]:
def resolve_selected_dir(chooser: FileChooser, fallback: Path) -> Path:
    selected = getattr(chooser, 'selected', None)
    if selected:
        return Path(selected).expanduser().resolve()
    return fallback.expanduser().resolve()

datasets_dir = resolve_selected_dir(datasets_dir_chooser, DEFAULT_DATASETS_DIR)
training_data_root = resolve_selected_dir(training_data_dir_chooser, DEFAULT_TRAINING_DATA_DIR)
training_data_root.mkdir(parents=True, exist_ok=True)

display(Markdown(
    f'- `datasets_dir`: `{datasets_dir}`\n'
    f'- `training_data_root`: `{training_data_root}`\n'
    f'- `PROCESS`: `{PROCESS}`'
))


In [ ]:
def infer_machine_name(dataset_dir_name: str) -> str:
    parts = dataset_dir_name.split('-')
    if len(parts) >= 3:
        return parts[1]
    return 'unknown'

def is_not_in_shards(path: Path, root: Path) -> bool:
    rel_parts = path.relative_to(root).parts
    return 'shards' not in rel_parts

if not datasets_dir.exists():
    raise FileNotFoundError(f'datasets_dir does not exist: {datasets_dir}')
if not training_data_root.exists():
    raise FileNotFoundError(f'training_data_root does not exist: {training_data_root}')

training_dir_names = {
    path.name
    for path in training_data_root.rglob('*')
    if path.is_dir() and is_not_in_shards(path, training_data_root)
}

dataset_dirs = sorted(path for path in datasets_dir.iterdir() if path.is_dir())
if TARGET_DATASET_DIR_NAMES is not None:
    target_set = {name.strip() for name in TARGET_DATASET_DIR_NAMES}
    dataset_dirs = [path for path in dataset_dirs if path.name in target_set]

scan_rows = []
processing_jobs = []
planned_files_rows = []

for dataset_dir in dataset_dirs:
    all_wavs = sorted(
        path
        for path in dataset_dir.rglob('*')
        if path.is_file() and path.suffix.lower() == '.wav'
    )
    if MAX_FILES_PER_DIRECTORY is None:
        selected_wavs = all_wavs
    else:
        selected_wavs = all_wavs[:MAX_FILES_PER_DIRECTORY]

    is_matched = dataset_dir.name in training_dir_names
    machine_name = infer_machine_name(dataset_dir.name)
    target_output_root = training_data_root / machine_name / dataset_dir.name

    scan_rows.append({
        'dataset_dir_name': dataset_dir.name,
        'dataset_dir': str(dataset_dir),
        'machine_name': machine_name,
        'is_matched_in_training_data': is_matched,
        'num_wav_files_total': len(all_wavs),
        'num_wav_files_selected': len(selected_wavs),
        'target_output_root': str(target_output_root),
    })

    if (not is_matched) and selected_wavs:
        processing_jobs.append({
            'dataset_dir_name': dataset_dir.name,
            'dataset_dir': dataset_dir,
            'machine_name': machine_name,
            'output_root': target_output_root,
            'source_files': selected_wavs,
            'num_source_files': len(selected_wavs),
        })

        for wav_path in selected_wavs:
            relative_source = wav_path.relative_to(dataset_dir)
            planned_npz_path = (target_output_root / 'tensors' / relative_source).with_suffix('.npz')
            planned_files_rows.append({
                'dataset_dir_name': dataset_dir.name,
                'machine_name': machine_name,
                'source_wav_path': str(wav_path),
                'planned_npz_path': str(planned_npz_path),
            })

scan_df = pd.DataFrame(scan_rows).sort_values('dataset_dir_name').reset_index(drop=True)
processing_plan_df = pd.DataFrame([
    {
        'dataset_dir_name': job['dataset_dir_name'],
        'machine_name': job['machine_name'],
        'num_source_files': job['num_source_files'],
        'output_root': str(job['output_root']),
    }
    for job in processing_jobs
]).sort_values('dataset_dir_name').reset_index(drop=True) if processing_jobs else pd.DataFrame(columns=['dataset_dir_name', 'machine_name', 'num_source_files', 'output_root'])
dry_run_df = pd.DataFrame(planned_files_rows)

display(Markdown('### Dataset Directory Scan'))
display(scan_df)

display(Markdown('### Unmatched Directories Selected For Export'))
display(processing_plan_df)

display(Markdown(
    f'- Dataset directories scanned: `{len(dataset_dirs)}`\n'
    f'- Matched directories skipped: `{int(scan_df["is_matched_in_training_data"].sum()) if not scan_df.empty else 0}`\n'
    f'- Unmatched directories selected: `{len(processing_jobs)}`\n'
    f'- Total selected WAV files: `{len(dry_run_df)}`'
))

if not processing_jobs:
    display(Markdown('No unmatched dataset directories with WAV files were found.'))


In [ ]:
export_jobs = []
for job in processing_jobs:
    config = ExportConfig(
        input_dir=job['dataset_dir'],
        output_root=job['output_root'],
        backend=BACKEND,
        source_files=job['source_files'],
        max_files=None,
        skip_existing=SKIP_EXISTING,
        target_sr=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        window=WINDOW,
        num_bands=NUM_BANDS,
        fmin=FMIN,
        fmax=FMAX,
        log_power_reference=LOG_POWER_REFERENCE,
        epsilon=EPSILON,
        normalize_band_energy=True,
        normalization_mode='per_clip_minmax',
        energy_bins=ENERGY_BINS,
        low_energy_floor=LOW_ENERGY_FLOOR,
        min_active_bands_per_frame=MIN_ACTIVE_BANDS_PER_FRAME,
        window_frames=WINDOW_FRAMES,
        stride_frames=STRIDE_FRAMES,
    )
    export_jobs.append({
        'dataset_dir_name': job['dataset_dir_name'],
        'machine_name': job['machine_name'],
        'dataset_dir': job['dataset_dir'],
        'output_root': job['output_root'],
        'config': config,
        'num_source_files': job['num_source_files'],
    })

display(Markdown('### Export Job Configuration'))
display(pd.DataFrame([
    {
        'dataset_dir_name': job['dataset_dir_name'],
        'machine_name': job['machine_name'],
        'num_source_files': job['num_source_files'],
        'output_root': str(job['output_root']),
        'backend': BACKEND,
        'skip_existing': SKIP_EXISTING,
        'process': PROCESS,
    }
    for job in export_jobs
]))


In [ ]:
run_summaries = []
run_summaries_df = pd.DataFrame()

if not PROCESS:
    display(Markdown('### Dry Run Mode\n`PROCESS = False`, so no files were exported.'))
elif not export_jobs:
    display(Markdown('### Nothing To Export\nNo unmatched dataset directories were selected for processing.'))
else:
    for index, job in enumerate(export_jobs, start=1):
        display(Markdown(
            f'#### [{index}/{len(export_jobs)}] Export `{job["dataset_dir_name"]}` ' 
            f'-> `{job["output_root"]}`'
        ))
        summary = export_directory(job['config'])
        summary = dict(summary)
        summary['dataset_dir_name'] = job['dataset_dir_name']
        summary['machine_name'] = job['machine_name']
        run_summaries.append(summary)

    run_summaries_df = pd.DataFrame(run_summaries)
    display(Markdown('### Run Summaries'))
    display(run_summaries_df)


In [ ]:
if not PROCESS:
    display(Markdown('### Dry-Run Report (Planned Exports)'))
    display(pd.DataFrame([{
        'num_dataset_dirs_scanned': len(dataset_dirs),
        'num_unmatched_dirs_selected': len(processing_jobs),
        'num_planned_wav_files': len(dry_run_df),
    }]))

    if dry_run_df.empty:
        display(Markdown('No files are currently planned for export.'))
    else:
        display(dry_run_df)
elif not run_summaries:
    display(Markdown('No export summaries were produced.'))
else:
    files_manifest_frames = []
    windows_manifest_frames = []

    for summary in run_summaries:
        files_manifest_path = Path(summary['files_manifest_path'])
        windows_manifest_path = Path(summary['windows_manifest_path'])

        files_df = pd.read_parquet(files_manifest_path)
        files_df['dataset_dir_name'] = summary['dataset_dir_name']
        files_df['machine_name'] = summary['machine_name']
        files_manifest_frames.append(files_df)

        windows_df = pd.read_parquet(windows_manifest_path)
        windows_df['dataset_dir_name'] = summary['dataset_dir_name']
        windows_df['machine_name'] = summary['machine_name']
        windows_manifest_frames.append(windows_df)

    combined_files_df = pd.concat(files_manifest_frames, ignore_index=True) if files_manifest_frames else pd.DataFrame()
    combined_windows_df = pd.concat(windows_manifest_frames, ignore_index=True) if windows_manifest_frames else pd.DataFrame()

    display(Markdown('### Combined Run Summary'))
    display(run_summaries_df)

    display(Markdown('### Files Manifest (sample)'))
    display(combined_files_df.head(20))

    display(Markdown('### Windows Manifest (sample)'))
    display(combined_windows_df.head(20))


## Smoke-Test Tips
- Keep `PROCESS = False` to preview what would run.
- Set `TARGET_DATASET_DIR_NAMES = ['+6db-fan-normal']` to focus on one dataset directory.
- Set `MAX_FILES_PER_DIRECTORY = 2` for a quick end-to-end validation before full export.
